In [3]:
import os
import time
import re
import pandas as pd
from bs4 import BeautifulSoup

# SSL 인증서 에러를 방지하기 위해 환경 변수 강제 초기화
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['REQUESTS_CA_BUNDLE'] = ''

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

def scrape_mmth_selenium():
    sido_list = ["서울", "경기", "인천", "강원", "충북", "충남", "대전", "세종", 
                 "전북", "전남", "광주", "경북", "경남", "대구", "울산", "부산", "제주"]
    
    # 브라우저 설정 (창을 띄우지 않으려면 headless 추가)
    chrome_options = Options()
    # chrome_options.add_argument("--headless") # 주석 해제 시 창이 뜨지 않음
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    all_stores = []
    print("매머드 커피 지점 정보 수집을 시작합니다. (Selenium 모드)\n")

    try:
        for sido in sido_list:
            start_time = time.time()
            url = f"https://mmthcoffee.com/sub/store.html?sido={sido}#content"
            
            driver.get(url)
            time.sleep(2) # 페이지 로딩 대기 (네트워크 상황에 따라 1~3초 조절)

            # 현재 렌더링된 페이지의 HTML을 가져와 BeautifulSoup으로 분석
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            
            # 선택자를 더 유연하게 가져옵니다. (txt_w 클래스가 있는 모든 div)
            store_items = soup.select('.txt_w')
            
            current_sido_count = 0
            for item in store_items:
                # 1. 지점ID 추출 (부모 a 태그의 href 찾기)
                parent_a = item.find_parent('a')
                store_id = ""
                if parent_a:
                    href_val = parent_a.get('href', '')
                    match = re.search(r'\((\d+)\)', href_val)
                    store_id = match.group(1) if match else ""
                
                # 2. 매점명 (tit 클래스)
                name_tag = item.select_one('.tit strong')
                name = name_tag.get_text(strip=True) if name_tag else ""
                
                # 3. 주소 (txt 클래스 내부의 p 태그)
                address_tag = item.select_one('.txt p')
                address = address_tag.get_text(strip=True).replace("주소 :", "").strip() if address_tag else ""
                
                if name:
                    all_stores.append({
                        "지역": sido,
                        "지점ID": store_id,
                        "매점명": name,
                        "주소": address
                    })
                    current_sido_count += 1

            end_time = time.time()
            print(f"[{sido}] 수집 완료 - 소요 시간: {end_time - start_time:.2f}초 (수집된 지점 수: {current_sido_count})")

    finally:
        driver.quit() # 브라우저 종료

    # CSV 저장
    if all_stores:
        df = pd.DataFrame(all_stores)
        df.to_csv("mmthcoffee.csv", index=False, encoding='utf-8-sig')
        print(f"\n총 {len(df)}개의 지점 정보를 'mmthcoffee.csv'로 저장했습니다.")
    else:
        print("\n수집된 데이터가 없습니다. 사이트 구조나 대기 시간을 확인해 보세요.")

if __name__ == "__main__":
    scrape_mmth_selenium()

매머드 커피 지점 정보 수집을 시작합니다. (Selenium 모드)

[서울] 수집 완료 - 소요 시간: 6.93초 (수집된 지점 수: 480)
[경기] 수집 완료 - 소요 시간: 3.22초 (수집된 지점 수: 233)
[인천] 수집 완료 - 소요 시간: 2.71초 (수집된 지점 수: 37)
[강원] 수집 완료 - 소요 시간: 2.62초 (수집된 지점 수: 4)
[충북] 수집 완료 - 소요 시간: 2.83초 (수집된 지점 수: 1)
[충남] 수집 완료 - 소요 시간: 2.64초 (수집된 지점 수: 4)
[대전] 수집 완료 - 소요 시간: 2.62초 (수집된 지점 수: 8)
[세종] 수집 완료 - 소요 시간: 2.55초 (수집된 지점 수: 2)
[전북] 수집 완료 - 소요 시간: 2.59초 (수집된 지점 수: 8)
[전남] 수집 완료 - 소요 시간: 2.65초 (수집된 지점 수: 5)
[광주] 수집 완료 - 소요 시간: 2.60초 (수집된 지점 수: 12)
[경북] 수집 완료 - 소요 시간: 2.60초 (수집된 지점 수: 7)
[경남] 수집 완료 - 소요 시간: 2.69초 (수집된 지점 수: 18)
[대구] 수집 완료 - 소요 시간: 2.65초 (수집된 지점 수: 10)
[울산] 수집 완료 - 소요 시간: 2.66초 (수집된 지점 수: 2)
[부산] 수집 완료 - 소요 시간: 2.62초 (수집된 지점 수: 13)
[제주] 수집 완료 - 소요 시간: 2.60초 (수집된 지점 수: 2)

총 846개의 지점 정보를 'mmthcoffee.csv'로 저장했습니다.


In [2]:
import pandas as pd
from sqlalchemy import create_engine, types
import os

def insert_to_mysql():
    file_path = "mmthcoffee.csv"
    
    # 1. CSV 파일 존재 확인
    if not os.path.exists(file_path):
        print(f"'{file_path}' 파일이 없습니다. 수집을 먼저 완료해주세요.")
        return

    # 2. 데이터 불러오기
    df = pd.read_csv(file_path)

    # 3. MySQL 연결 설정 (본인의 환경에 맞게 수정하세요)
    # 형식: mysql+pymysql://아이디:비밀번호@호스트:포트/데이터베이스명
    user = "root"          # MySQL 아이디
    password = "1234"  # MySQL 비밀번호
    host = "localhost"     # 호스트 (보통 localhost)
    port = "3306"          # 포트 (보통 3306)
    db_name = "coffee_store"

    try:
        # 데이터베이스 연결 엔진 생성
        engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}?charset=utf8mb4")

        # 4. 데이터 타입을 지정하여 테이블 생성 및 삽입
        # 지점ID는 숫자로, 나머지는 문자열(VARCHAR/TEXT)로 지정합니다.
        dtype_dict = {
            '지역': types.VARCHAR(50),
            '지점ID': types.Integer(),
            '매점명': types.VARCHAR(100),
            '주소': types.TEXT()
        }

        # 5. DB에 데이터 넣기
        # if_exists='replace': 이미 테이블이 있으면 삭제하고 새로 만듭니다.
        # index=False: 판다스 인덱스는 컬럼으로 넣지 않습니다.
        df.to_sql(name='mmthcoffee', con=engine, if_exists='replace', index=False, dtype=dtype_dict)

        print(f"성공! '{db_name}' DB의 'mmthcoffee' 테이블에 총 {len(df)}건의 데이터가 삽입되었습니다.")

    except Exception as e:
        print(f"DB 접속 중 에러 발생: {e}")
        print("참고: 'coffee_store' 데이터베이스가 미리 생성되어 있는지 확인해주세요.")

if __name__ == "__main__":
    insert_to_mysql()

성공! 'coffee_store' DB의 'mmthcoffee' 테이블에 총 846건의 데이터가 삽입되었습니다.
